In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
from chaotic_data import systems # more details about this can be found in a separate folder at the root of the repo
from utils import preprocess , plottings #, custom_fcts

In [ ]:
# lorenz_data, _ = systems.LorenzSolver(start=0.0, stop=20.0, ics=(1.0, 0.0, 1.25), time_grid=0.01)
# lorenz_data.shape  # Expected shape: (2000, 3)

In [ ]:
rabi_data, _= systems.RabinovichFabrikantSolver(start=0.0, stop=200.0, ics=(0.1 , 0.1 ,0.1), time_grid=0.1)
rabi_data.shape

In [ ]:
plotter_lorenz  =plottings.ChaoticSystemPlotter(data=rabi_data, system_name='Lorenz')
plotter_lorenz.plot_all_trajectories(render='horizontal')



In [ ]:
norma = preprocess.Normalization_Strategy()   
scaled_rabi, avg_rabi = norma.Scale_ToNormalize(data=rabi_data, scaling_factor=0.02)
print(scaled_rabi.shape , avg_rabi.shape)  # 

In [ ]:
train_size = 1500
train_rabi = scaled_rabi[:train_size]
test_rabi = scaled_rabi[train_size:]
print(train_rabi.shape, test_rabi.shape)

In [ ]:
data_sliding = preprocess.SlidingWindowDataset(data=train_rabi, window_size=200)
sliced_data = data_sliding.sliding_windows_shift_to(train_rabi, 200)

print(sliced_data.shape)   

In [ ]:
batch_size = 20
my_data_loader = preprocess.SlidingWindowDataLoader(dataset=sliced_data, batch_size=batch_size, shuffle=False)

In [ ]:
calibration_window_length =200
forecast_window_length = 100

context_data_r, toforecast_data_r = data_sliding.contextwindow_testdata_generator(
    context_size=calibration_window_length, fcast_size=forecast_window_length, location='random_'
)


context_data_r.shape , toforecast_data_r.shape

In [ ]:
from HCNN.ensembles import VanillaHCNNEnsembleTrainer, PTFHCNNEnsembleTrainer , HCNNLFormEnsembleTrainer, LSPaEnsembleTrainer

# Ensemble of Vanilla HCNN Models

In [ ]:
trainer = VanillaHCNNEnsembleTrainer(
    n_ensemble=5,
    n_obs_vars=3,
    n_hid_vars=20,
    s0_nature='random_',
    train_s0=True,
    optimizer="adam",
    loss_fn="logcosh",
    lr=1e-4,
    best_on='median')



In [ ]:
trainer.train_only(data_loader= my_data_loader, epochs=4 )

In [ ]:
trainer.save_loss_history("loss_log.csv")


In [ ]:
trainer.load_all_best()  # reload best models automatically

forecasts = trainer.forecast(context_data=context_data_r, forecast_window_length=forecast_window_length)

In [ ]:
forecasts = trainer.forecast(context_data=context_data_r, forecast_window_length=forecast_window_length)

In [ ]:
plot_together = plottings.ChaoticSystemPlotter(data = toforecast_data_r, system_name='Lorenz')

In [ ]:
ensemble_tensors = torch.stack(forecasts, dim=0)
# ensemble_tensors = ensemble_tensors[:,:100,:]
ensemble_tensors.shape

In [ ]:
plot_together.plot_ensemble(ensemble_tensors, toforecast_data_r, add_mean_or_median='median' , render="horizontal")

In [ ]:
import pandas as pd

In [ ]:
pd.read_csv("loss_log.csv")

# Ensemble of HCNNLSpa models

In [ ]:
lspa_trainer = LSPaEnsembleTrainer(
    n_ensemble=5,
    n_obs_vars=3,
    n_hid_vars=20,
    s0_nature='random_', sparsity_ratio=0.25, mask_type="non_obs_block",
    optimizer="adam",lr=1e-4,loss_fn="mse",
    best_on="median")

In [ ]:
data_sliding_init = preprocess.SlidingWindowDataset(data=train_rabi, window_size=1)
# data_sliding

In [ ]:
calibration_window_length =1
forecast_window_length = 1499

context_data_r, toforecast_data_r = data_sliding_init.contextwindow_testdata_generator(
    context_size=calibration_window_length, fcast_size=forecast_window_length, location='random_'
)


context_data_r.shape , toforecast_data_r.shape

In [ ]:
sliced_data = data_sliding_init.sliding_windows_shift_to(train_rabi, 1500)

print(sliced_data.shape)   

In [ ]:
batch_size = 1
my_second_data_loader = preprocess.SlidingWindowDataLoader(dataset=sliced_data, batch_size=batch_size, shuffle=False)


In [ ]:
lspa_trainer.train_and_validate( my_second_data_loader, calibration_data=context_data_r, validation_data=toforecast_data_r,
                            epochs=4 )

In [ ]:
lspa_trainer.save_loss_history("loss_log.csv")
lspa_trainer.load_all_best()  # reload best models automatically

forecasts = trainer.forecast(context_data=context_data_r, forecast_window_length=forecast_window_length)
plot_together = plottings.ChaoticSystemPlotter(data = toforecast_data_r, system_name='Lorenz')
ensemble_tensors = torch.stack(forecasts, dim=0)
# ensemble_tensors = ensemble_tensors[:,:100,:]
ensemble_tensors.shape

In [ ]:
plot_together.plot_ensemble(ensemble_tensors, toforecast_data_r, add_mean_or_median='median' , render="horizontal")

# Ensemble of HCNN-pTF

In [ ]:
trainer_ptf = PTFHCNNEnsembleTrainer( n_ensemble=5,
                                     n_obs_vars=3, n_hid_vars=5, s0_nature='zeros_',train_s0=True,
                                        target_prob=0.35, drop_output=False,optimizer="adam",
                                        loss_fn="mse", lr=1e-4, best_on="median")

In [ ]:
trainer_ptf.train_only(my_data_loader, epochs=4 )# or "individual"

# Ensemble of HCNN-LForm

In [ ]:
lform_trainer = HCNNLFormEnsembleTrainer(n_ensemble=4, n_obs_vars=3, n_hid_vars=6, s0_nature="zeros_",
                                         train_s0=True, optimizer="adam", loss_fn="mse", lr=1e-4,
                                         init_range=(-0.75, 0.75), init_diag=1.0, best_on="median")

In [ ]:
lform_trainer.train_and_validate(data_loader=my_data_loader, 
                                 calibration_data=context_data_r, 
                                 validation_data=toforecast_data_r, 
                                 epochs=4)  # or "individual"

In [ ]:
lform_trainer.load_all_best()  # reload best models automatically

forecasts = lform_trainer.forecast(context_data=context_data_r, forecast_window_length=forecast_window_length)

In [ ]:
plot_together = plottings.ChaoticSystemPlotter(data = toforecast_data_r, system_name='Lorenz')
ensemble_tensors = torch.stack(forecasts, dim=0)
# ensemble_tensors = ensemble_tensors[:,:100,:]
ensemble_tensors.shape

In [ ]:

plot_together.plot_ensemble(ensemble_tensors, toforecast_data_r, add_mean_or_median='median' , render="horizontal")